# Programming Assignment 3 — Poucos Rótulos: Fauna em Câmeras Trap com Dados Não Rotulados

**Disciplina:** Aprendizado Profundo  
**Professor:** Dario Oliveira | **Monitor:** Erick Brito  
**Alunos:** Henrique Gabriel Gasparelo & José Thevez Gomes Guedes  

---

### Objetivo do Projeto
Construir um classificador de espécies em imagens de câmeras trap (*ENA24-detection*, 21 classes) em um regime severo de escassez de dados anotados (**5 rótulos por classe**, totalizando 105 imagens rotuladas) e milhares de imagens não rotuladas (**6.505 imagens**).

### Restrições de Engenharia
1. **Teto de Parâmetros:** Máximo de **12 milhões de parâmetros** (`sum(p.numel() for p in model.parameters()) <= 12_000_000`).
2. **Ambiente:** Modelo leve executável em **CPU**, sem internet, classificando o conjunto de teste em menos de 10 minutos.
3. **Métrica Principal:** **F1 Macro** no conjunto de teste, equilibrando o peso entre classes abundantes (ex: urso) e raras (ex: cavalo).

## 1. Setup, Imports e Fixação de Seeds

In [ ]:
import os
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
from torchvision import models
from sklearn.metrics import f1_score, accuracy_score, confusion_matrix, classification_report

# Fixação estrita de sementes para reprodutibilidade
def seed_everything(seed: int = 42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

seed_everything(42)

# Dispositivo de processamento para desenvolvimento (inferência final roda em CPU)
device = torch.device('cuda' if torch.cuda.is_available() else ('mps' if torch.backends.mps.is_available() else 'cpu'))
print(f"Dispositivo de treinamento ativo: {device}")

## 2. Verificação de Regras de Engenharia (Teto de 12M Parâmetros)

In [ ]:
def check_parameter_limit(model: nn.Module, max_params: int = 12_000_000) -> bool:
    """Verifica se a arquitetura atende à restrição de no máximo 12 milhões de parâmetros."""
    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"Total de parâmetros: {total_params:,} (Treináveis: {trainable_params:,})")
    if total_params <= max_params:
        print(f"✓ Aprovado: O modelo cumpre a restrição de <= {max_params:,} parâmetros.")
        return True
    else:
        print(f"✗ Reprovado: O modelo excede o limite permitido ({total_params:,} > {max_params:,})!")
        return False

# Teste com ResNet-18 padrão
test_resnet = models.resnet18(weights=None, num_classes=21)
check_parameter_limit(test_resnet)

## 3. Exploração dos Dados (ENA24)

In [ ]:
DATA_DIR = "data"

# Verificação da presença dos metadados descompactados
expected_files = ["rotulos_1.csv", "rotulos_5.csv", "rotulos_20.csv", "val.csv", "test_public.csv", "classes.csv"]
found_files = [f for f in expected_files if os.path.exists(os.path.join(DATA_DIR, f))]

print(f"Arquivos encontrados em '{DATA_DIR}': {found_files}")
if len(found_files) < len(expected_files):
    print("Aviso: Certifique-se de baixar e extrair o arquivo 'pa3_dados.zip' na pasta 'data/'.")

## 4. Parte 0 — O Piso (Baseline Supervisionado com 105 Imagens)

- **Versão (a):** Pequena CNN treinada do zero com apenas as 105 imagens de `rotulos_5.csv`.
- **Versão (b):** Backbone pré-treinado no ImageNet (ex: ResNet-18) adaptado com *fine-tuning*.
- **Objetivo:** Registrar o F1 Macro e Acurácia no `test_public`, matriz de confusão e curvas de *overfitting*.

In [ ]:
# [Planejado] Implementação da Parte 0:
# - Dataset/DataLoader para rotulos_5.csv, val.csv e test_public.csv
# - Treinamento da versão (a) [CNN do zero] e versão (b) [Backbone ImageNet]
# - Curvas de Loss e F1 Macro Treino vs Validação
# - Avaliação no test_public e Matriz de Confusão

## 5. Parte 1 — Usando o que Não Tem Rótulo (Trilhas A, B ou C)

Exploração do conjunto com as 6.505 imagens não rotuladas do `train` sob uma das estratégias autorizadas:
- **Trilha A:** Auto-supervisionado (SimCLR ou SwAV) + avaliação linear e fine-tuning.
- **Trilha B:** Semi-supervisionado (FixMatch com correção de viés de classes desbalanceadas).
- **Trilha C:** Professor & Aluno (Destilação de conhecimento de Foundation Model aberto).

In [ ]:
# [Planejado] Implementação da Trilha escolhida da Parte 1:
# - Pré-treino ou geração de pseudo-rótulos sobre as 6.505 imagens não rotuladas
# - Fine-tuning com os 105 rótulos de rotulos_5.csv
# - Comparação com o piso supervisionado

## 6. Parte 2 — A Curva de Orçamento

- Orçamentos avaliados: **1, 5 e 20 rótulos por classe**.
- No orçamento de 5 rótulos: **3 seeds** para cálculo de média $\pm$ desvio padrão.
- Gráfico semi-log: F1 Macro no `test_public` vs. número de rótulos com a linha do teto de referência (0,686).
- Exportação para `results/curva_orcamento.csv`.

In [ ]:
# [Planejado] Execução das ablações de orçamento:
# - Treino nos orçamentos 1, 5 (3 seeds) e 20
# - Tabela de resultados consolidados
# - Gráfico comparativo: Piso vs Método vs Teto (0.686)

## 7. Parte 3 — Galeria de Falhas e Atalhos (Saliência / Grad-CAM)

Inspeção interpretável das decisões do modelo final:
- **3 Falhas:** Imagem, classe verdadeira, classe predita, confiança e mapa de calor do Grad-CAM.
- **5 Acertos:** Análise se a predição foi motivada pelo animal ou por atalhos espúrios do fundo.

In [ ]:
# [Planejado] Grad-CAM e visualização de saliência:
# - Extração de mapas de ativação na última camada convolucional
# - Plotagem lado a lado da imagem original, mapa de calor e sobreposição

## 8. Exportação do Modelo e Teste de Inferência CPU (`predict.py`)

In [ ]:
# [Planejado] Salvamento dos pesos em checkpoints/best_model.pt
# e verificação da inferência isolada em CPU